In [1]:
#!pip install sympy

In [2]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from qhdopt import QHD

In [3]:
# 初始化（用默认 3-bus 数据）
#model = SympyACOPFModel()   # 打开 proximal 选项，后面会用到

# 2bus model
Sbase_2 = 10.0
buses_2 = {
    1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
    2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
}
lines_2 = {
    1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase_2],
}
gens_2 = {
    1: [1, 0.0 / Sbase_2, 20.0 / Sbase_2, -20.0 / Sbase_2, 100.0 / Sbase_2, 0.00375, 2.0, 0.0],
}

#model = SympyACOPFModel(Sbase = Sbase_2, buses=buses_2, lines=lines_2, gens=gens_2)
model = SympyACOPFModel() # 打开 proximal 选项，后面会用到

In [4]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 5.0
alpha = 5e-3   # 对偶步长尽量小一点
max_outer = 200
tol = 1e-4

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=10.0
        )

    option = 1
    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        qhd_solver = "simbi" # openjij / simbi
        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=7,
                agents=2048,
                max_steps=3000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                verbose=True
            )
        else:
            qhd_model.openjij_setup(
                resolution=6,
                shots=1024,
                sampler_name="SQASampler",
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 3000,
                    "reinitialize_state": True,
                },
            )
        response = qhd_model.optimize(refine=True, verbose=0)

        x_new = np.asarray(response.refined_minimizer)

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    PrintQHDACOPFResults(model, x_new)

    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1024
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (7) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (6) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")


===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:   0%|          | 0/2048 [00:01<?, ?it/s]


backend time consumption: 1.651249647140503
Minimizer: [0.85714286 0.71428571 0.71428571 0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.71428571 0.71428571 0.85714286 0.71428571
 0.71428571 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.71428571
 0.85714286 0.71428571 0.71428571 0.71428571 0.71428571 0.71428571
 0.71428571]
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0331	0.0087	0.0118	0.0354	1.0000	0.0000	0.0000	0.0000
2	1.0333	0.0081	0.0139	0.0182	1.0000	0.0000	0.0000	0.0000
3	1.0211	-0.0044	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.0257	0.0536		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	-0.0021	0.0145	0.0052	0.0101
1	3	0.0283	-0.0270	0.0073	-0.0175
2	3	0.0485	-0.0277	0.0151	-0.0152


Total Ploss: 0.0345
Total Qloss: 0.0049
Total Load Supplied: -2.9140%
||h(x)|| = 0.22231451187916923
[rho-check] ||h_old||=5.109e-01, ||h_new||=2.223e-01, rho=5
Constraint check: False

--- Outer Iteration 1 ---


Bifurcated agents:   0%|          | 0/2048 [00:01<?, ?it/s]


backend time consumption: 1.5653128623962402
Minimizer: [0.71428571 0.71428571 0.71428571 0.85714286 0.85714286 0.85714286
 0.71428571 0.71428571 0.71428571 0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.71428571 0.71428571 0.85714286
 0.71428571 0.71428571 0.71428571 0.85714286 0.71428571 0.71428571
 0.71428571]
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	1.0484	0.0511	0.0000	0.0126	1.0000	0.0000	0.0000	0.0000
2	1.0488	0.0515	0.0000	0.0171	1.0000	0.0000	0.0000	0.0000
3	1.0408	0.0406	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.0000	0.0297		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	-0.0021	0.0047	-0.0067	0.0102
1	3	0.0591	-0.0564	0.0255	-0.0285
2	3	0.0745	-0.0636	0.0259	-0.0259


Total Ploss: 0.0162
Total Qloss: 0.0005
Total Load Supplied: -5.3920%
||h(x)|| = 0.20057702683649775
[rho-check] ||h_old||=2.223e-01, ||h_new||=2.006e-01, rho=5
Increasing rho to 10.0
Constraint check: False

--- Out

...

Bifurcated agents:   0%|          | 0/2048 [00:01<?, ?it/s]


backend time consumption: 1.61769700050354
Minimizer: [0.85714286 0.85714286 0.71428571 0.71428571 0.85714286 0.85714286
 0.71428571 0.71428571 0.71428571 0.85714286 0.85714286 0.71428571
 0.85714286 0.85714286 0.85714286 0.85714286 0.71428571 0.85714286
 0.85714286 0.71428571 0.85714286 0.71428571 0.85714286 0.71428571
 0.71428571 0.85714286 0.71428571 0.71428571 0.71428571 0.85714286
 0.71428571]
BusID	VR	VI	Pg	Qg	l	Pl	Ql	Qshunt

1	0.8353	0.0799	0.1843	-2.0000	1.0000	0.0000	0.0000	0.0000
2	0.9428	0.0487	0.1241	2.1780	1.0000	0.0000	0.0000	0.0000
3	0.8867	0.0506	0.0000	0.0000	1.0000	0.3000	0.1000	0.0000


TOTAL			0.3084	0.1780		0.3000	0.1000



Busi	Busk	Pik	Pki	Qik	Qki
1	2	0.1112	0.0241	-1.8254	1.8962
1	3	0.0890	-0.0822	-0.2543	0.2900
2	3	0.0555	-0.1226	0.3646	-0.3279


Total Ploss: 0.0751
Total Qloss: 0.1432
Total Load Supplied: 77.7801%
||h(x)|| = 0.4922965584409594
[rho-check] ||h_old||=8.502e-01, ||h_new||=4.923e-01, rho=1.28e+03
Constraint check: False

--- Outer Iteration 50 ---

Bifurcated agents:   0%|          | 0/2048 [00:01<?, ?it/s]


backend time consumption: 1.555424690246582
Minimizer: [0.85714286 0.71428571 0.85714286 0.71428571 0.85714286 0.85714286
 0.71428571 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.71428571 0.85714286 0.71428571 0.85714286
 0.85714286 0.71428571 0.85714286 0.85714286 1.         0.71428571
 0.71428571 0.85714286 0.85714286 0.71428571 0.71428571 0.71428571
 0.71428571]


KeyboardInterrupt: 

In [ ]:
response.refined_minimizer

In [ ]:
x_new

In [ ]:
Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=np.ones_like(xk),
            rho=rho,
            ref_bus_id=None,
            mu_prox=1500.0
        )

In [ ]:
Lag

In [ ]:
model.linear_jacobian

In [ ]:
model.G_mat

In [ ]:
model.arc_collection

In [ ]:
response.refined_minimizer


In [ ]:
qhd_model.func

In [ ]:
print(model.P_D, model.Q_D, model.P_G, model.Q_G)
print(model.variable_list)
print(model.Var_bound_list)